### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="amex_non_iid_1m",
    version_from_unique_name="amex_non_iid",
    version_comment="""
We build the recommended single train/test split (250k test rows) and sub-sample it to at most 1M train and 250k test rows with `curation_recommendations.subsample_split_to_budget`, keeping whole customers.
""",
    # Same as the full-size `amex_non_iid` dataset
    dataset_year="2022",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/amex-default-prediction/",
    download_description="""
We start with the preprocessed version of the dataset in Parquet format provided by
the user 'raddar' on Kaggle. At the root of this project, I ran on linux with
the kaggle CLI installed and authenticated:

kaggle datasets download -d raddar/amex-data-integer-dtypes-parquet-format --unzip -p amex_dataset && rm amex_dataset/test.parquet
kaggle competitions download -c amex-default-prediction -f train_labels.csv -p amex_dataset && cd amex_dataset && unzip train_labels.csv.zip train_labels.csv && rm train_labels.csv.zip && cd ..
mkdir -p local-data-warehouse/amex_non_iid && mv amex_dataset/* local-data-warehouse/amex_non_iid/ && rm -rf amex_dataset
""",
    # References
    academic_reference_bibtex="""@misc{howard2022amex,
  author       = {Howard, Addison and AritraAmex and Xu, Di and Vashani, Hossein and inversion and Negin and Dane, Sohier},
  title        = {American Express -- Default Prediction},
  year         = {2022},
  howpublished = {Kaggle Competition},
  url          = {https://kaggle.com/competitions/amex-default-prediction},
  note         = {Accessed via Kaggle}
}
""",
    academic_reference_bibtex_key="howard2022amex",
    license="Kaggle Competition License",
    data_tags=["Non-IID", "Grouped", "Anonymized"],
    curation_comments="""
We start with the raw data from Kaggle and do not apply any further preprocessing to simulate a pipeline that can handle raw non-IID grouped data.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="amex_metric",
    stratify_on="target",
    group_on="customer_ID",
    group_time_on="S_2",
    group_labels="per_group",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "train.parquet")
print("Loaded data shape:", df.shape)

df.S_2 = pd.to_datetime(df.S_2)
df = df.sort_values(["customer_ID", "S_2"])

cat_features = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
    "target",
]
df = df.set_index("customer_ID")
# Add labels
targets = pd.read_csv(dataset_mold.path / "train_labels.csv")
targets = targets.set_index("customer_ID")
df = df.merge(targets, left_index=True, right_index=True, how="left")
df.target = df.target.astype("int8")
del targets

df[cat_features] = df[cat_features].astype("category")

df = df.reset_index().reset_index(drop=True)
df["customer_ID"] = df["customer_ID"].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False,
    duplicate_column_check=False,
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

# Sub-sample the single train/test split to the row budget (1M train / 250k test rows), keeping whole customers.
train_idx, test_idx = splits[0][0]
df, train_idx, test_idx = curation_recommendations.subsample_split_to_budget(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    group_on=task_mold.group_on,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)
splits = {0: {0: (train_idx, test_idx)}}
print(f"Final train size: {len(train_idx):,}, final test size: {len(test_idx):,}")
print(f"Final train groups: {df.iloc[train_idx][task_mold.group_on].nunique():,}, final test groups: {df.iloc[test_idx][task_mold.group_on].nunique():,}")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Recommended single grouped train/test split, sub-sampled to the row budget of 1M train and 250k test rows, keeping whole customers.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()